# EstateMind — Local Agentic Pipeline

**Input**: `data/bigfinal_realestate_Cleaned.csv` (264,847 × 73, produced by `bigfinal_preprocessing.ipynb`)

**What this notebook does**

1. Trains 6 price-prediction models (XGBoost, LightGBM, RandomForest, CatBoost, PyTorch-MLP, Stacking)
2. Selects the best, adds SHAP explanations, saves it to `artifacts/models/best_price_model.pkl`
3. Downloads a free GGUF LLM (Mistral-7B-Instruct Q4_K_M by default)
4. Wraps the model + tools in a ReAct agent (Thought -> Action -> Observation loop)
5. Runs 3 end-to-end example queries
6. Generates a PDF report `artifacts/reports/report_realestate_YYYY-MM-DD.pdf`

All paths are **relative to the project root** — launch Jupyter from there.


## Section 0 — Setup

### 0.1 · Install packages

Skips anything already installed. On RTX 50-series you may need to install PyTorch and llama-cpp-python with CUDA wheels manually (see `requirements_agent.txt`).

In [ ]:
# ── Section 0.1 — Install packages ─────────────────────────────────
import subprocess, sys, importlib

def _ensure(spec: str, import_name: str | None = None):
    import_name = import_name or spec.split("==")[0].split(">=")[0]
    try:
        importlib.import_module(import_name)
    except Exception:
        print(f"[pip] installing {spec}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", spec])

for spec, mod in [
    ("pandas>=2.2", "pandas"), ("numpy>=1.26", "numpy"),
    ("scikit-learn>=1.4", "sklearn"), ("xgboost>=2.0", "xgboost"),
    ("lightgbm>=4.3", "lightgbm"), ("catboost>=1.2", "catboost"),
    ("shap>=0.46", "shap"), ("matplotlib>=3.8", "matplotlib"),
    ("seaborn>=0.13", "seaborn"), ("plotly>=5.22", "plotly"),
    ("tqdm>=4.66", "tqdm"), ("joblib>=1.4", "joblib"),
    ("reportlab>=4.2", "reportlab"), ("fpdf2>=2.7", "fpdf"),
    ("huggingface_hub>=0.24", "huggingface_hub"),
]:
    _ensure(spec, mod)

# torch and llama-cpp-python — skip auto-install (need CUDA-specific wheels).
# The notebook falls back to CPU or to a deterministic-mock agent if absent.
for mod in ["torch", "llama_cpp"]:
    try:
        importlib.import_module(mod)
        print(f"[ok] {mod} available")
    except Exception:
        print(f"[warn] {mod} not installed — see requirements_agent.txt for CUDA-specific install")


### 0.2 · Imports + CUDA detection

In [ ]:
# ── Section 0.2 — Imports + CUDA ───────────────────────────────────
import os, sys, json, time, math, re, warnings, random, pickle, datetime as dt
from pathlib import Path
from typing import Any
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns

warnings.filterwarnings("ignore")
np.random.seed(42); random.seed(42)

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"  GPU: {torch.cuda.get_device_name(0)}  | capability sm_{''.join(map(str, torch.cuda.get_device_capability(0)))}")
except Exception as e:
    torch = None
    DEVICE = "cpu"
    print(f"[warn] torch not available: {e}")

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_theme(style="whitegrid")
print(f"Device selected: {DEVICE}")


### 0.3 · Paths (relative to project root)

In [ ]:
# ── Section 0.3 — Paths ────────────────────────────────────────────
# Resolve project root by walking up until we find the data file.
def _find_root() -> Path:
    cur = Path.cwd().resolve()
    for p in [cur] + list(cur.parents):
        if (p / "data" / "bigfinal_realestate_Cleaned.csv").exists():
            return p
    return cur

ROOT = _find_root()
DATA_CSV  = ROOT / "data" / "bigfinal_realestate_Cleaned.csv"
MODELS_DIR = ROOT / "artifacts" / "models";   MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = ROOT / "artifacts" / "reports"; REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGS_DIR    = ROOT / "artifacts" / "agent_figs"; FIGS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_GGUF_DIR = ROOT / "artifacts" / "llm"; MODEL_GGUF_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT       :", ROOT)
print("DATA_CSV   :", DATA_CSV, "| exists:", DATA_CSV.exists())
print("MODELS_DIR :", MODELS_DIR)
print("REPORTS_DIR:", REPORTS_DIR)


## Section 1 — Load cleaned dataset

In [ ]:
# ── Section 1.1 — Load + quick look ────────────────────────────────
assert DATA_CSV.exists(), f"Missing {DATA_CSV}. Run bigfinal_preprocessing.ipynb first."
df = pd.read_csv(DATA_CSV, low_memory=False)
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
print("Target stats:", df["prix"].describe(percentiles=[0.05, 0.5, 0.95]).to_dict())
df.head(3)


### 1.2 · Filter to valid rows for price modelling

In [ ]:
# ── Section 1.2 — Filter rows with valid price + surface ───────────
before = len(df)
m = (
    df["prix"].between(5_000, 50_000_000)
    & df["surface"].between(10, 10_000)
)
df_model = df.loc[m].copy()
print(f"Kept {len(df_model):,} / {before:,} rows for training (valid prix + surface)")


## Section 2 — Feature engineering

**Leakage-safe feature set** (excludes anything derived from `prix`):
- Numeric: surface, pieces, salle_de_bain, etage, latitude, longitude, annee_constr, superficie_terrain, bus, railway, pub_year, pub_month
- Derived (leakage-free): haut_standing, bon_entourage
- Binary: all `has_*` columns
- Categorical: type, gouvernerat, contrat


In [ ]:
# ── Section 2.1 — Define features ──────────────────────────────────
NUM_COLS = [
    "surface", "pieces", "salle_de_bain", "etage",
    "latitude", "longitude", "annee_constr", "superficie_terrain",
    "bus", "railway", "pub_year", "pub_month",
    "haut_standing", "bon_entourage",
]
HAS_COLS = [c for c in df_model.columns if c.startswith("has_")]
CAT_COLS = ["type", "gouvernerat", "contrat"]

FEATURES = [c for c in NUM_COLS + HAS_COLS + CAT_COLS if c in df_model.columns]
TARGET   = "prix"

print(f"Using {len(FEATURES)} features ({len(NUM_COLS)} numeric + {len(HAS_COLS)} binary + {len(CAT_COLS)} categorical)")

# Force-coerce every supposedly-numeric column: some still contain strings like '1 SDB'.
for c in NUM_COLS + HAS_COLS:
    if c in df_model.columns:
        df_model[c] = pd.to_numeric(df_model[c], errors="coerce")

X = df_model[FEATURES].copy()
y = df_model[TARGET].astype(float).copy()

# Log-transform target for stability (heavy-tailed prices).
y_log = np.log1p(y)
print("Target (log1p) stats:", y_log.describe().to_dict())


### 2.2 · Preprocessor + 70/15/15 split

In [ ]:
# ── Section 2.2 — Preprocessor + split ─────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUM_USED = [c for c in NUM_COLS + HAS_COLS if c in FEATURES]
CAT_USED = [c for c in CAT_COLS if c in FEATURES]

pre_numeric = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc",  StandardScaler()),
])
pre_cat = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("oh",  OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preproc = ColumnTransformer([
    ("num", pre_numeric, NUM_USED),
    ("cat", pre_cat,     CAT_USED),
])

# 70 / 15 / 15
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y_log, test_size=0.15, random_state=42)
X_tr,  X_val,  y_tr,  y_val  = train_test_split(X_tmp, y_tmp, test_size=0.1765, random_state=42)
print(f"train={len(X_tr):,}  val={len(X_val):,}  test={len(X_test):,}")

preproc.fit(X_tr)
Xtr_p = preproc.transform(X_tr)
Xva_p = preproc.transform(X_val)
Xte_p = preproc.transform(X_test)
print("Preprocessed train matrix:", Xtr_p.shape)


### 2.3 · Shared evaluation helper

One function -> MAE / RMSE / R² / MAPE on the original (un-logged) price scale, consistent across every model.

In [ ]:
# ── Section 2.3 — Metrics helper ───────────────────────────────────
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def eval_regression(y_true_log: np.ndarray, y_pred_log: np.ndarray, label: str) -> dict:
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2  = r2_score(y_true, y_pred)
    mape = float(np.mean(np.abs((y_true - y_pred) / np.clip(y_true, 1, None))) * 100)
    return {"model": label, "MAE": mae, "RMSE": rmse, "R2": r2, "MAPE%": mape}

SCORES: list[dict] = []
MODELS: dict[str, Any] = {}


## Section 3 — Price Arena

Six models trained on the **same** train/val/test split. Each has an architecture blurb, then training code, then evaluation.

### 3.1 · XGBoost

Gradient-boosted decision trees with histogram-based splits.
- **Objective**: `reg:squarederror` on `log1p(prix)`
- **Trees**: 800 max, early stopping on val
- **Depth**: 8, learning rate 0.05, subsample 0.8, colsample 0.8
- **Device**: CUDA if available (`device="cuda"`), else CPU


In [ ]:
# ── Section 3.1 — XGBoost ──────────────────────────────────────────
import xgboost as xgb

xgb_params = dict(
    n_estimators=800, max_depth=8, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    tree_method="hist",
    device="cuda" if (torch is not None and torch.cuda.is_available()) else "cpu",
    random_state=42, n_jobs=-1,
)
m_xgb = xgb.XGBRegressor(**xgb_params, early_stopping_rounds=30)
m_xgb.fit(Xtr_p, y_tr, eval_set=[(Xva_p, y_val)], verbose=False)
pred = m_xgb.predict(Xte_p)
s = eval_regression(y_test.values, pred, "XGBoost")
SCORES.append(s); MODELS["XGBoost"] = m_xgb
print(s)


### 3.2 · LightGBM

Leaf-wise gradient boosting with GOSS sampling.
- **Leaves**: 127, learning rate 0.05, min_child_samples 50
- **n_estimators**: 1000 with early stopping


In [ ]:
# ── Section 3.2 — LightGBM ─────────────────────────────────────────
import lightgbm as lgb

m_lgb = lgb.LGBMRegressor(
    n_estimators=1000, num_leaves=127, learning_rate=0.05,
    min_child_samples=50, subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbosity=-1,
)
m_lgb.fit(Xtr_p, y_tr, eval_set=[(Xva_p, y_val)],
          callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
pred = m_lgb.predict(Xte_p)
s = eval_regression(y_test.values, pred, "LightGBM")
SCORES.append(s); MODELS["LightGBM"] = m_lgb
print(s)


### 3.3 · Random Forest

Bagged CART forest, 400 trees, max_depth 24, min_samples_leaf 5.

In [ ]:
# ── Section 3.3 — Random Forest ────────────────────────────────────
from sklearn.ensemble import RandomForestRegressor

m_rf = RandomForestRegressor(
    n_estimators=400, max_depth=24, min_samples_leaf=5,
    n_jobs=-1, random_state=42,
)
m_rf.fit(Xtr_p, y_tr)
pred = m_rf.predict(Xte_p)
s = eval_regression(y_test.values, pred, "RandomForest")
SCORES.append(s); MODELS["RandomForest"] = m_rf
print(s)


### 3.4 · CatBoost

Ordered-boosting, native categorical handling (we feed pre-encoded matrix anyway for fairness).
- **Iterations**: 1500, depth 8, learning rate 0.05
- **Task_type**: GPU if CUDA, else CPU


In [ ]:
# ── Section 3.4 — CatBoost ─────────────────────────────────────────
from catboost import CatBoostRegressor

cat_task = "GPU" if (torch is not None and torch.cuda.is_available()) else "CPU"
m_cat = CatBoostRegressor(
    iterations=1500, depth=8, learning_rate=0.05,
    loss_function="RMSE", task_type=cat_task,
    random_seed=42, verbose=False, early_stopping_rounds=50,
)
try:
    m_cat.fit(Xtr_p, y_tr, eval_set=(Xva_p, y_val), verbose=False)
except Exception as e:
    # GPU sometimes fails on unsupported shapes — fall back to CPU.
    print(f"[catboost] GPU failed ({e}); retrying on CPU")
    m_cat = CatBoostRegressor(iterations=1500, depth=8, learning_rate=0.05,
                              task_type="CPU", random_seed=42, verbose=False,
                              early_stopping_rounds=50)
    m_cat.fit(Xtr_p, y_tr, eval_set=(Xva_p, y_val), verbose=False)
pred = m_cat.predict(Xte_p)
s = eval_regression(y_test.values, pred, "CatBoost")
SCORES.append(s); MODELS["CatBoost"] = m_cat
print(s)


### 3.5 · PyTorch MLP (3 hidden layers)

```
Input(D) -> Linear(256) -> ReLU -> Dropout(0.2)
        -> Linear(128) -> ReLU -> Dropout(0.2)
        -> Linear(64)  -> ReLU
        -> Linear(1)
```
- Loss: HuberLoss (robust to outliers even on log-scale)
- Optimizer: AdamW(lr=1e-3, weight_decay=1e-4)
- Epochs: 30 with early stopping patience 5 on val loss
- Batch: 2048 on GPU, 512 on CPU


In [ ]:
# ── Section 3.5 — PyTorch MLP ──────────────────────────────────────
class _MLP:
    '''Thin wrapper that looks like a sklearn regressor to keep arena uniform.'''
    def __init__(self, in_dim: int, device: str = "cpu"):
        import torch.nn as nn
        self.device = device
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 128),    nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64),     nn.ReLU(),
            nn.Linear(64, 1),
        ).to(device)

    def fit(self, X, y, X_val=None, y_val=None, epochs=30, bs=2048, lr=1e-3, patience=5):
        import torch
        from torch.utils.data import DataLoader, TensorDataset
        Xt = torch.tensor(np.asarray(X, dtype=np.float32), device=self.device)
        yt = torch.tensor(np.asarray(y, dtype=np.float32), device=self.device).unsqueeze(1)
        loader = DataLoader(TensorDataset(Xt, yt), batch_size=bs, shuffle=True)
        opt  = torch.optim.AdamW(self.net.parameters(), lr=lr, weight_decay=1e-4)
        loss_fn = torch.nn.HuberLoss(delta=0.5)
        best_val, bad, best_state = float("inf"), 0, None
        for ep in range(epochs):
            self.net.train()
            tot = 0.0
            for xb, yb in loader:
                opt.zero_grad()
                loss = loss_fn(self.net(xb), yb)
                loss.backward(); opt.step()
                tot += float(loss) * len(xb)
            if X_val is not None:
                vl = self._val_loss(X_val, y_val, loss_fn)
                if vl < best_val - 1e-4:
                    best_val, bad, best_state = vl, 0, {k: v.detach().clone() for k, v in self.net.state_dict().items()}
                else:
                    bad += 1
                    if bad >= patience: break
        if best_state is not None:
            self.net.load_state_dict(best_state)
        return self

    def _val_loss(self, X, y, loss_fn):
        import torch
        self.net.eval()
        with torch.no_grad():
            Xt = torch.tensor(np.asarray(X, dtype=np.float32), device=self.device)
            yt = torch.tensor(np.asarray(y, dtype=np.float32), device=self.device).unsqueeze(1)
            return float(loss_fn(self.net(Xt), yt))

    def predict(self, X):
        import torch
        self.net.eval()
        with torch.no_grad():
            Xt = torch.tensor(np.asarray(X, dtype=np.float32), device=self.device)
            return self.net(Xt).cpu().numpy().ravel()


if torch is not None:
    bs = 2048 if DEVICE == "cuda" else 512
    m_mlp = _MLP(Xtr_p.shape[1], device=DEVICE).fit(Xtr_p, y_tr.values, Xva_p, y_val.values, bs=bs)
    pred  = m_mlp.predict(Xte_p)
    s = eval_regression(y_test.values, pred, "PyTorchMLP")
    SCORES.append(s); MODELS["PyTorchMLP"] = m_mlp
    print(s)
else:
    print("[skip] torch unavailable — MLP not trained")


### 3.6 · Stacking ensemble

Meta-learner: **Ridge**. Base learners: XGBoost + LightGBM + RandomForest.
Out-of-fold predictions from 5-fold CV on the train set feed the meta.

In [ ]:
# ── Section 3.6 — Stacking ─────────────────────────────────────────
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge

base = [
    ("xgb", xgb.XGBRegressor(n_estimators=400, max_depth=7, learning_rate=0.05,
                             tree_method="hist", n_jobs=-1, random_state=42,
                             device="cuda" if (torch is not None and torch.cuda.is_available()) else "cpu")),
    ("lgb", lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=63,
                              n_jobs=-1, random_state=42, verbosity=-1)),
    ("rf",  RandomForestRegressor(n_estimators=200, max_depth=20, n_jobs=-1, random_state=42)),
]
m_stack = StackingRegressor(estimators=base, final_estimator=Ridge(alpha=1.0),
                            n_jobs=1, passthrough=False, cv=5)
m_stack.fit(Xtr_p, y_tr)
pred = m_stack.predict(Xte_p)
s = eval_regression(y_test.values, pred, "Stacking")
SCORES.append(s); MODELS["Stacking"] = m_stack
print(s)


### 3.7 · 5-fold cross-validation (sanity check)

Each model's R² across 5 folds on a random 40K-row subset — confirms the test-set ranking isn't a fluke.

In [ ]:
# ── Section 3.7 — CV sanity check ──────────────────────────────────
from sklearn.model_selection import KFold, cross_val_score

_n = min(40_000, len(X_tr))
idx = np.random.RandomState(42).choice(len(X_tr), _n, replace=False)
Xs, ys = Xtr_p[idx], y_tr.values[idx]
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_rows = []
for name, mdl in [("XGBoost", m_xgb), ("LightGBM", m_lgb),
                  ("RandomForest", m_rf), ("CatBoost", m_cat)]:
    try:
        sc = cross_val_score(mdl, Xs, ys, cv=kf, scoring="r2", n_jobs=1)
        cv_rows.append({"model": name, "cv_R2_mean": float(sc.mean()), "cv_R2_std": float(sc.std())})
        print(f"{name:14s} R² = {sc.mean():.4f} ± {sc.std():.4f}")
    except Exception as e:
        print(f"[skip] {name}: {e}")

pd.DataFrame(cv_rows)


### 3.8 · Final comparison table + plots

In [ ]:
# ── Section 3.8 — Arena leaderboard + plots ────────────────────────
leaderboard = pd.DataFrame(SCORES).sort_values("MAE").reset_index(drop=True)
print("\n=== Price Arena Leaderboard (sorted by MAE on test set) ===")
display(leaderboard.round(3))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=leaderboard, x="model", y="MAE", ax=ax[0], palette="rocket")
ax[0].set_title("MAE by model (lower = better)"); ax[0].tick_params(axis="x", rotation=30)
sns.barplot(data=leaderboard, x="model", y="R2", ax=ax[1], palette="mako")
ax[1].set_title("R² by model (higher = better)"); ax[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig(FIGS_DIR / "arena_leaderboard.png", dpi=140, bbox_inches="tight")
plt.show()

BEST_NAME  = leaderboard.iloc[0]["model"]
BEST_MODEL = MODELS[BEST_NAME]
print(f"\n>>> Best model: {BEST_NAME}")


### 3.9 · Actual vs Predicted + residuals (best model)

In [ ]:
# ── Section 3.9 — Diagnostic plots for best model ──────────────────
pred_best = BEST_MODEL.predict(Xte_p)
y_true = np.expm1(y_test.values)
y_hat  = np.expm1(pred_best)
resid  = y_true - y_hat

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].scatter(y_true, y_hat, s=4, alpha=0.3)
lim = [0, np.percentile(y_true, 99)]
ax[0].plot(lim, lim, "r--", lw=1)
ax[0].set_xlim(lim); ax[0].set_ylim(lim)
ax[0].set_xlabel("Actual (DT)"); ax[0].set_ylabel("Predicted (DT)")
ax[0].set_title(f"Actual vs Predicted — {BEST_NAME}")

sns.histplot(resid, bins=80, ax=ax[1])
ax[1].axvline(0, color="r", lw=1)
ax[1].set_xlim(np.percentile(resid, [1, 99]))
ax[1].set_title("Residual distribution"); ax[1].set_xlabel("Residual (DT)")
plt.tight_layout()
plt.savefig(FIGS_DIR / "best_model_diagnostics.png", dpi=140, bbox_inches="tight")
plt.show()


### 3.10 · Feature importance (best tree model)

In [ ]:
# ── Section 3.10 — Feature importance ──────────────────────────────
num_names = NUM_USED
try:
    cat_names = list(preproc.named_transformers_["cat"].named_steps["oh"].get_feature_names_out(CAT_USED))
except Exception:
    cat_names = [f"cat_{i}" for i in range(Xtr_p.shape[1] - len(num_names))]
FEAT_NAMES = num_names + cat_names

def _importances(mdl):
    if hasattr(mdl, "feature_importances_"):
        return np.asarray(mdl.feature_importances_, dtype=float)
    return None

imp = _importances(BEST_MODEL)
if imp is None and BEST_NAME == "Stacking":
    imp = _importances(BEST_MODEL.named_estimators_["xgb"])

if imp is not None:
    imp_df = (pd.DataFrame({"feature": FEAT_NAMES[:len(imp)], "importance": imp})
                .sort_values("importance", ascending=False).head(20))
    plt.figure(figsize=(10, 7))
    sns.barplot(data=imp_df, y="feature", x="importance", palette="viridis")
    plt.title(f"Top-20 feature importances — {BEST_NAME}")
    plt.tight_layout()
    plt.savefig(FIGS_DIR / "feature_importance.png", dpi=140, bbox_inches="tight")
    plt.show()
else:
    print(f"[info] {BEST_NAME} exposes no feature_importances_ — skipping plot")


### 3.11 · Best model justification

The arena-leader is selected on **test-set MAE** (primary metric — interpretable in dinars). The comparison plot in 3.8 shows where each model ranks on both MAE and R². The SHAP section next breaks down how the best model uses individual features for a specific prediction.

### 3.12 · SHAP explanations

In [ ]:
# ── Section 3.12 — SHAP for best model ─────────────────────────────
import shap

try:
    if BEST_NAME in {"XGBoost", "LightGBM", "CatBoost", "RandomForest"}:
        explainer = shap.TreeExplainer(BEST_MODEL)
    elif BEST_NAME == "Stacking":
        explainer = shap.TreeExplainer(BEST_MODEL.named_estimators_["xgb"])
    else:
        # Kernel explainer on a tiny sample (slow but model-agnostic)
        explainer = shap.KernelExplainer(BEST_MODEL.predict, shap.sample(Xtr_p, 100))

    sample = Xte_p[:200] if hasattr(Xte_p, "__getitem__") else Xte_p[:200]
    sv = explainer.shap_values(sample)

    shap.summary_plot(sv, sample, feature_names=FEAT_NAMES[:sample.shape[1]],
                      max_display=15, show=False)
    plt.tight_layout()
    plt.savefig(FIGS_DIR / "shap_summary.png", dpi=140, bbox_inches="tight")
    plt.show()
    SHAP_READY = True
except Exception as e:
    print(f"[warn] SHAP failed: {e}")
    explainer, SHAP_READY = None, False


### 3.13 · Persist best model + preprocessor

In [ ]:
# ── Section 3.13 — Save artifacts ──────────────────────────────────
import joblib

bundle = {
    "name": BEST_NAME,
    "model": BEST_MODEL,
    "preproc": preproc,
    "features": FEATURES,
    "num_used": NUM_USED,
    "cat_used": CAT_USED,
    "feat_names": FEAT_NAMES,
    "trained_at": dt.datetime.now().isoformat(timespec="seconds"),
    "leaderboard": leaderboard.to_dict(orient="records"),
}
bundle_path = MODELS_DIR / "best_price_model.pkl"
joblib.dump(bundle, bundle_path, compress=3)
print(f"Saved bundle -> {bundle_path}  ({bundle_path.stat().st_size/1e6:.1f} MB)")


## Section 4 — ReAct Agent

### Architecture

```
┌──────────────────────────────────────────────────────────────┐
│                     User natural-language query              │
└──────────────────────────────┬───────────────────────────────┘
                               ▼
                 ┌──────────────────────────────┐
                 │   Local LLM (llama-cpp /     │
                 │   Mistral-7B-Instruct Q4_K_M)│
                 │   generates: THOUGHT -> ACTION│
                 └──────────────┬───────────────┘
                                │   ACTION = tool name + JSON args
                                ▼
   ┌────────────┬──────────────┬─────────────┬──────────────┬────────────┐
   │predict_    │recommend_    │market_      │explain_      │generate_   │
   │price       │properties    │summary      │prediction    │report      │
   └────────────┴──────────────┴─────────────┴──────────────┴────────────┘
                                │   OBSERVATION (JSON/str)
                                ▼
                 ┌──────────────────────────────┐
                 │  LLM ingests OBSERVATION ->   │
                 │  either another THOUGHT/     │
                 │  ACTION or FINAL ANSWER      │
                 └──────────────┬───────────────┘
                                ▼
                           Final answer
```

The ReAct loop is capped at **6 iterations** — well above the 1–3 most queries need.

### 4.1 · Download a free GGUF model

In [ ]:
# ── Section 4.1 — Download GGUF ────────────────────────────────────
from huggingface_hub import hf_hub_download

# Primary: Mistral-7B-Instruct. Fallback: Qwen2.5-7B-Instruct.
GGUF_CANDIDATES = [
    {"repo": "TheBloke/Mistral-7B-Instruct-v0.2-GGUF",
     "file": "mistral-7b-instruct-v0.2.Q4_K_M.gguf",
     "chat_format": "mistral-instruct",
     "label": "Mistral-7B-Instruct-v0.2-Q4_K_M"},
    {"repo": "Qwen/Qwen2.5-7B-Instruct-GGUF",
     "file": "qwen2.5-7b-instruct-q4_k_m.gguf",
     "chat_format": "chatml",
     "label": "Qwen2.5-7B-Instruct-Q4_K_M"},
]

GGUF_PATH: Path | None = None
GGUF_META: dict | None = None
for spec in GGUF_CANDIDATES:
    try:
        print(f"[hf] trying {spec['repo']}/{spec['file']} …")
        path = hf_hub_download(repo_id=spec["repo"], filename=spec["file"],
                               local_dir=MODEL_GGUF_DIR, local_dir_use_symlinks=False)
        GGUF_PATH, GGUF_META = Path(path), spec
        print(f"[hf] ok -> {GGUF_PATH}")
        break
    except Exception as e:
        print(f"[hf] {spec['repo']} failed: {e}")

if GGUF_PATH is None:
    print("[warn] no GGUF model could be downloaded — agent will fall back to a deterministic mock")


### 4.2 · Load the model in llama-cpp-python

In [ ]:
# ── Section 4.2 — Load LLM ─────────────────────────────────────────
LLM = None
LLM_LABEL = "mock"
try:
    if GGUF_PATH is not None:
        from llama_cpp import Llama
        n_gpu = -1 if (torch is not None and torch.cuda.is_available()) else 0
        LLM = Llama(
            model_path=str(GGUF_PATH),
            n_ctx=4096, n_gpu_layers=n_gpu,
            chat_format=GGUF_META["chat_format"],
            verbose=False, seed=42,
        )
        LLM_LABEL = GGUF_META["label"]
        print(f"[llm] loaded {LLM_LABEL}  (n_gpu_layers={n_gpu})")
    else:
        raise RuntimeError("no GGUF available")
except Exception as e:
    print(f"[warn] LLM load failed ({e}) — using deterministic mock agent")
    LLM = None


### 4.3 · Tool implementations

Each tool accepts a dict of kwargs and returns a JSON-serialisable object. The agent sees only their name + docstring in the system prompt.

In [ ]:
# ── Section 4.3 — Tool: predict_price ──────────────────────────────
def _row_from_features(features: dict) -> pd.DataFrame:
    '''Project a user-supplied feature dict onto the trained FEATURES list.
    Missing fields -> NaN (SimpleImputer handles it).'''
    row = {c: features.get(c, np.nan) for c in FEATURES}
    return pd.DataFrame([row], columns=FEATURES)

def tool_predict_price(features: dict) -> dict:
    '''Predict price (DT) for a property described by `features`. Keys can include
    surface, pieces, salle_de_bain, type, gouvernerat, contrat, has_jardin, ... .'''
    row = _row_from_features(features)
    Xp  = preproc.transform(row)
    log_pred = float(BEST_MODEL.predict(Xp)[0])
    price = float(np.expm1(log_pred))
    # Rough ±MAE band from leaderboard
    mae = float(leaderboard.iloc[0]["MAE"])
    return {
        "predicted_price_TND": round(price, 0),
        "confidence_band_TND": [round(max(0, price - mae), 0), round(price + mae, 0)],
        "model": BEST_NAME,
    }


In [ ]:
# ── Section 4.3 — Tool: recommend_properties ───────────────────────
def tool_recommend_properties(budget: float, preferences: dict | None = None, top_k: int = 5) -> list[dict]:
    '''Return top_k listings under `budget` matching `preferences`
    (region=gouvernerat, type, min_pieces, min_surface, needs = list of has_* flags).'''
    prefs = dict(preferences or {})
    d = df_model.copy()
    d = d[d["prix"] <= float(budget)]
    if (r := prefs.get("region")): d = d[d["gouvernerat"].astype(str).str.contains(str(r), case=False, na=False)]
    if (t := prefs.get("type")):   d = d[d["type"].astype(str).str.contains(str(t), case=False, na=False)]
    if (p := prefs.get("min_pieces")):   d = d[d["pieces"] >= float(p)]
    if (s := prefs.get("min_surface")):  d = d[d["surface"] >= float(s)]
    for flag in (prefs.get("needs") or []):
        if flag in d.columns:
            d = d[d[flag] == 1]
    d = d.sort_values(["prix", "surface"], ascending=[True, False]).head(int(top_k))
    cols = ["prix", "surface", "pieces", "type", "gouvernerat", "contrat", "url"]
    cols = [c for c in cols if c in d.columns]
    return d[cols].fillna("").to_dict(orient="records")


In [ ]:
# ── Section 4.3 — Tool: market_summary ─────────────────────────────
def tool_market_summary(region: str) -> dict:
    '''Return mean/median price, count, 12-month trend slope for a gouvernerat.'''
    d = df_model[df_model["gouvernerat"].astype(str).str.contains(str(region), case=False, na=False)].copy()
    if d.empty:
        return {"region": region, "error": "no listings found"}
    out = {
        "region":       region,
        "n_listings":   int(len(d)),
        "mean_price":   float(d["prix"].mean()),
        "median_price": float(d["prix"].median()),
        "p25_price":    float(d["prix"].quantile(0.25)),
        "p75_price":    float(d["prix"].quantile(0.75)),
        "mean_prix_m2": float(d["prix_m2"].mean()) if "prix_m2" in d.columns else None,
    }
    # Simple 12-month slope on monthly medians if pub_year/month populated
    if "pub_year" in d.columns and d["pub_year"].notna().any():
        ts = (d.dropna(subset=["pub_year", "pub_month"])
                .assign(_ym=lambda x: x["pub_year"].astype(int).astype(str) + "-" + x["pub_month"].astype(int).astype(str).str.zfill(2))
                .groupby("_ym")["prix"].median().sort_index())
        if len(ts) >= 3:
            slope = float(np.polyfit(np.arange(len(ts)), ts.values, 1)[0])
            out["trend_slope_TND_per_month"] = slope
            out["trend"] = "rising" if slope > 0 else "falling"
    # Anomalies = > mean + 3*std
    thresh = d["prix"].mean() + 3 * d["prix"].std()
    out["n_outliers_above_3sigma"] = int((d["prix"] > thresh).sum())
    return out


In [ ]:
# ── Section 4.3 — Tool: explain_prediction ─────────────────────────
def tool_explain_prediction(features: dict, top_k: int = 6) -> dict:
    '''Return SHAP-ranked contributions (feature, shap_value) for a single property.'''
    row = _row_from_features(features)
    Xp  = preproc.transform(row)
    if not SHAP_READY or explainer is None:
        # fallback -> use feature_importances_ × standardised feature values
        imp = getattr(BEST_MODEL, "feature_importances_", None)
        if imp is None:
            return {"error": "no explainer available"}
        contribs = [(FEAT_NAMES[i], float(imp[i]) * float(Xp[0, i])) for i in range(len(imp))]
        contribs.sort(key=lambda t: abs(t[1]), reverse=True)
        return {"method": "importance×value", "top_contributions": contribs[:top_k]}
    sv = explainer.shap_values(Xp)
    vec = sv[0] if getattr(sv, "ndim", 2) > 1 else sv
    contribs = sorted(zip(FEAT_NAMES[:len(vec)], map(float, vec)), key=lambda t: abs(t[1]), reverse=True)[:top_k]
    return {
        "method": "SHAP",
        "top_contributions": contribs,
        "baseline_log_price": float(getattr(explainer, "expected_value", 0)),
    }


In [ ]:
# ── Section 4.3 — Tool: generate_report (PDF) ──────────────────────
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
                                Image as RLImage, PageBreak)
from reportlab.lib.units import cm

def _style():
    ss = getSampleStyleSheet()
    ss.add(ParagraphStyle(name="H1x", parent=ss["Heading1"], textColor=colors.HexColor("#1a5276"),
                          spaceAfter=12))
    ss.add(ParagraphStyle(name="H2x", parent=ss["Heading2"], textColor=colors.HexColor("#2874a6")))
    return ss

def tool_generate_report(query_results: dict) -> str:
    '''Write a PDF summarising the agent's session. Returns the file path.'''
    ss = _style()
    today = dt.date.today().isoformat()
    out_path = REPORTS_DIR / f"report_realestate_{today}.pdf"

    doc = SimpleDocTemplate(str(out_path), pagesize=A4, title="EstateMind — Session Report")
    story = []

    # Title page ---------------------------------------------------
    story.append(Paragraph("EstateMind", ss["H1x"]))
    story.append(Paragraph(f"Tunisian real-estate AI session report — {today}", ss["Normal"]))
    story.append(Spacer(1, 0.5*cm))
    story.append(Paragraph(f"<b>Best model</b>: {BEST_NAME}", ss["Normal"]))
    story.append(Paragraph(f"<b>LLM</b>: {LLM_LABEL}", ss["Normal"]))
    story.append(Paragraph(f"<b>Dataset</b>: {DATA_CSV.name} — {len(df_model):,} rows used for training", ss["Normal"]))
    story.append(Spacer(1, 0.5*cm))

    # Executive summary
    story.append(Paragraph("Executive summary", ss["H2x"]))
    story.append(Paragraph(query_results.get("executive_summary",
        "Agent answered a set of property-pricing queries and produced recommendations."),
        ss["Normal"]))
    story.append(Spacer(1, 0.4*cm))

    # Model comparison table
    story.append(Paragraph("Model comparison", ss["H2x"]))
    tbl_data = [["Model", "MAE (DT)", "RMSE (DT)", "R²", "MAPE %"]]
    for r in leaderboard.to_dict(orient="records"):
        tbl_data.append([r["model"], f"{r['MAE']:.0f}", f"{r['RMSE']:.0f}",
                         f"{r['R2']:.3f}", f"{r['MAPE%']:.1f}"])
    t = Table(tbl_data, hAlign="LEFT")
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1a5276")),
        ("TEXTCOLOR",  (0, 0), (-1, 0), colors.whitesmoke),
        ("FONTNAME",   (0, 0), (-1, 0), "Helvetica-Bold"),
        ("GRID",       (0, 0), (-1, -1), 0.5, colors.grey),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.whitesmoke, colors.white]),
    ]))
    story.append(t)
    story.append(PageBreak())

    # Recommendations
    recs = query_results.get("recommendations") or []
    if recs:
        story.append(Paragraph("Top property recommendations", ss["H2x"]))
        hdr = list(recs[0].keys())
        data = [hdr] + [[str(r.get(k, ""))[:60] for k in hdr] for r in recs[:5]]
        t = Table(data, hAlign="LEFT", repeatRows=1)
        t.setStyle(TableStyle([
            ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#239b56")),
            ("TEXTCOLOR",  (0, 0), (-1, 0), colors.whitesmoke),
            ("FONTSIZE",   (0, 0), (-1, -1), 7),
            ("GRID",       (0, 0), (-1, -1), 0.4, colors.grey),
        ]))
        story.append(t); story.append(Spacer(1, 0.4*cm))

    # Price prediction + SHAP chart
    if "prediction" in query_results:
        story.append(Paragraph("Price prediction", ss["H2x"]))
        pp = query_results["prediction"]
        story.append(Paragraph(str(pp), ss["Normal"]))
        shap_png = FIGS_DIR / "shap_summary.png"
        if shap_png.exists():
            story.append(RLImage(str(shap_png), width=16*cm, height=9*cm))
        story.append(Spacer(1, 0.4*cm))

    # Market charts + anomalies
    diag = FIGS_DIR / "best_model_diagnostics.png"
    if diag.exists():
        story.append(Paragraph("Model diagnostics", ss["H2x"]))
        story.append(RLImage(str(diag), width=16*cm, height=7*cm))
    if (market := query_results.get("market_summary")):
        story.append(Paragraph("Market summary", ss["H2x"]))
        story.append(Paragraph(str(market), ss["Normal"]))

    doc.build(story)
    return str(out_path)


### 4.4 · Register tools + ReAct loop

In [ ]:
# ── Section 4.4 — Tool registry + ReAct agent ──────────────────────
TOOL_REGISTRY: dict[str, dict] = {
    "predict_price": {
        "fn": tool_predict_price,
        "desc": "Predict a property price in TND. Args: features (dict of feature:value).",
    },
    "recommend_properties": {
        "fn": tool_recommend_properties,
        "desc": "Top-5 properties under a budget. Args: budget (float TND), preferences (dict with keys region, type, min_pieces, min_surface, needs=[list of has_* flags]).",
    },
    "market_summary": {
        "fn": tool_market_summary,
        "desc": "Market stats for a gouvernerat. Args: region (str).",
    },
    "explain_prediction": {
        "fn": tool_explain_prediction,
        "desc": "SHAP-style explanation for a predicted property. Args: features (dict).",
    },
    "generate_report": {
        "fn": tool_generate_report,
        "desc": "Writes the PDF report of the session. Args: query_results (dict).",
    },
}

_TOOL_LIST = "\n".join(f"- {n}: {v['desc']}" for n, v in TOOL_REGISTRY.items())
SYSTEM_PROMPT = f'''You are EstateMind, a Tunisian real-estate AI agent. You reason step-by-step and USE TOOLS.

Available tools:
{_TOOL_LIST}

You MUST answer using this exact format, one step at a time:
Thought: <your reasoning>
Action: <tool_name>
Action Input: <JSON dict with the arguments>

After you output Action Input, STOP and wait for the Observation. Then continue with another Thought/Action/Action Input, or emit:
Final Answer: <your final human-readable answer>

Rules:
- Action Input MUST be valid JSON (double-quoted keys, no trailing commas).
- Never invent tool names outside the list.
- Use at most 6 tool calls per query.
- When you have enough information, output Final Answer immediately.
'''


In [ ]:
# ── Section 4.4 — ReAct agent (continued) ──────────────────────────
_ACTION_RE  = re.compile(r"Action:\s*([a-zA-Z_]+)", re.IGNORECASE)
_FINAL_RE   = re.compile(r"Final Answer:\s*(.+)", re.IGNORECASE | re.DOTALL)

def _extract_action_input(text: str) -> str | None:
    '''Find 'Action Input:' then return the first balanced {...} block after it.'''
    m = re.search(r"Action Input:\s*", text, re.IGNORECASE)
    if not m:
        return None
    i = text.find("{", m.end())
    if i < 0:
        return None
    depth = 0
    in_str = False; esc = False
    for j in range(i, len(text)):
        ch = text[j]
        if esc: esc = False; continue
        if ch == "\\" and in_str: esc = True; continue
        if ch == '"': in_str = not in_str; continue
        if in_str: continue
        if ch == "{": depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[i:j+1]
    return None

class ReActAgent:
    def __init__(self, llm=None, max_iters: int = 6, verbose: bool = True):
        self.llm, self.max_iters, self.verbose = llm, max_iters, verbose

    def _chat(self, messages: list[dict]) -> str:
        '''One shot of the LLM. Falls back to a deterministic mock when self.llm is None.'''
        if self.llm is not None:
            resp = self.llm.create_chat_completion(
                messages=messages, temperature=0.1, max_tokens=512,
                stop=["Observation:", "\nObservation"],
            )
            return resp["choices"][0]["message"]["content"]
        # Mock: parse the user question for simple keywords and emit a reasonable action.
        q = messages[-1]["content"].lower()
        trace = "\n".join(m.get("content", "") for m in messages if m["role"] == "assistant")
        # Terminate after the first tool call so the mock doesn't loop forever.
        if "Action:" in trace:
            return 'Final Answer: Based on the tool output above, here is the answer for your query.'
        if "predict" in q or "price of" in q:
            return ('Thought: I should predict the price using the ML model.\n'
                    'Action: predict_price\n'
                    'Action Input: {"features": {"surface": 120, "pieces": 3, "gouvernerat": "Tunis", "type": "Maison", "has_jardin": 1, "has_parking": 1}}')
        if "recommend" in q or "find me" in q or "under" in q:
            return ('Thought: Recommend listings under the budget.\n'
                    'Action: recommend_properties\n'
                    'Action Input: {"budget": 300000, "preferences": {"region": "Tunis", "min_pieces": 3}}')
        if "market" in q or "trend" in q or "why" in q:
            return ('Thought: Market summary for the region.\n'
                    'Action: market_summary\n'
                    'Action Input: {"region": "Tunis"}')
        return 'Final Answer: I need more information.'

    def run(self, query: str) -> dict:
        history = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": query},
        ]
        trace: list[dict] = []
        for step in range(1, self.max_iters + 1):
            out = self._chat(history).strip()
            if self.verbose:
                print(f"── Step {step} ──\n{out}\n")
            if (m := _FINAL_RE.search(out)):
                final = m.group(1).strip()
                trace.append({"step": step, "llm": out, "final": final})
                return {"answer": final, "trace": trace, "iterations": step}
            act_m    = _ACTION_RE.search(out)
            inp_json = _extract_action_input(out)
            if not act_m or inp_json is None:
                # LLM went off-format -> ask it to retry.
                history.append({"role": "assistant", "content": out})
                history.append({"role": "user", "content":
                    "Your response must be 'Thought: ... / Action: <tool> / Action Input: {json}' OR 'Final Answer: ...'."})
                continue
            tool_name = act_m.group(1).strip()
            args: dict = {}
            try:
                parsed = json.loads(inp_json)
                if isinstance(parsed, dict):
                    args = parsed
            except json.JSONDecodeError as e:
                obs = f"JSON parse error: {e}. Please emit valid JSON."
            else:
                tool = TOOL_REGISTRY.get(tool_name)
                if tool is None:
                    obs = f"Unknown tool '{tool_name}'. Available: {list(TOOL_REGISTRY)}"
                else:
                    try:
                        obs = tool["fn"](**args)
                    except TypeError:
                        # tool expects a single positional dict
                        obs = tool["fn"](args)
                    except Exception as e:
                        obs = f"Tool error: {e}"
            obs_str = json.dumps(obs, default=str)[:1500]
            if self.verbose:
                print(f"Observation: {obs_str}\n")
            trace.append({"step": step, "tool": tool_name, "args": args if isinstance(args, dict) else {},
                          "observation": obs})
            history.append({"role": "assistant", "content": out})
            history.append({"role": "user", "content": f"Observation: {obs_str}"})
        return {"answer": "(max iterations reached)", "trace": trace,
                "iterations": self.max_iters}

agent = ReActAgent(llm=LLM, max_iters=6, verbose=True)
print(f"Agent ready — backend: {LLM_LABEL}")


## Section 5 — Three end-to-end agent runs

### 5.1 · Query 1 — "Find me a 3-bedroom apartment in Tunis under 300,000 DT and explain why prices in that area are high"

In [ ]:
# ── Section 5.1 — Query 1 ──────────────────────────────────────────
q1 = "Find me a 3-bedroom apartment in Tunis under 300000 DT and explain why prices in that area are high."
result1 = agent.run(q1)
print("\n=== FINAL ANSWER ===\n", result1["answer"])


### 5.2 · Query 2 — "Predict the price of a 120m² house in Ariana with a garden and parking"

In [ ]:
# ── Section 5.2 — Query 2 ──────────────────────────────────────────
q2 = "Predict the price of a 120m² house in Ariana with a garden and parking, 3 bedrooms."
result2 = agent.run(q2)
print("\n=== FINAL ANSWER ===\n", result2["answer"])


### 5.3 · Query 3 — "Compare the market between Tunis and Sfax and give me one outlier in each"

In [ ]:
# ── Section 5.3 — Query 3 ──────────────────────────────────────────
q3 = "Compare the market between Tunis and Sfax and tell me the trend in each."
result3 = agent.run(q3)
print("\n=== FINAL ANSWER ===\n", result3["answer"])


## Section 6 — PDF session report

In [ ]:
# ── Section 6.1 — Build PDF from the 3 runs ────────────────────────
exec_summary = (
    f"Agent ran {3} end-to-end queries using a {BEST_NAME} price model (test MAE ≈ "
    f"{leaderboard.iloc[0]['MAE']:.0f} DT, R² = {leaderboard.iloc[0]['R2']:.3f}) "
    f"and the {LLM_LABEL} LLM."
)

# Grab the most informative observation from each run.
def _extract(trace: list, tool: str):
    for step in trace:
        if step.get("tool") == tool:
            return step.get("observation")
    return None

recs   = _extract(result1["trace"], "recommend_properties") or []
market = _extract(result1["trace"], "market_summary") or _extract(result3["trace"], "market_summary")
pred   = _extract(result2["trace"], "predict_price") or {}

pdf_path = tool_generate_report({
    "executive_summary": exec_summary,
    "recommendations":   recs if isinstance(recs, list) else [],
    "market_summary":    market,
    "prediction":        pred,
    "final_answers": [result1["answer"], result2["answer"], result3["answer"]],
})
print(f"PDF written -> {pdf_path}")


## Section 7 — Launch the live dashboard

```bash
python dashboard.py
```

Opens at **http://localhost:8050**. The dashboard re-reads `data/bigfinal_realestate_Cleaned.csv` every 60 seconds and loads the best model from `artifacts/models/best_price_model.pkl` (produced in Section 3.13). The agent-chat panel streams Thought/Action/Observation steps identical to this notebook.

Required artifacts for the dashboard:
- `data/bigfinal_realestate_Cleaned.csv` — ✅ produced by preprocessing
- `artifacts/models/best_price_model.pkl` — ✅ produced in Section 3.13
- `artifacts/llm/*.gguf` — optional; dashboard falls back to mock agent if missing
